# 4.2 状態・メソッド・正しいオブジェクト

このNotebookでは、正しい生成、状態遷移、拒否後の状態保持を実行して確かめます。

In [ ]:
def required(value, label):
    cleaned = value.strip()
    if not cleaned:
        raise ValueError(f"{label} must not be empty")
    return cleaned

class EquipmentItem:
    def __init__(self, item_id, name, category, borrower_id=""):
        self.item_id = required(item_id, "item_id")
        self.name = required(name, "name")
        self.category = required(category, "category")
        self.borrower_id = borrower_id.strip() or None

    def is_available(self):
        return self.borrower_id is None

    def loan_to(self, borrower_id):
        borrower_id = required(borrower_id, "borrower_id")
        if not self.is_available():
            raise ValueError("item is already on loan")
        self.borrower_id = borrower_id

    def return_item(self):
        if self.is_available():
            raise ValueError("item is not on loan")
        self.borrower_id = None

    def to_record(self):
        return {"item_id": self.item_id, "name": self.name,
                "category": self.category, "borrower_id": self.borrower_id or ""}

## 独立した状態を確認する

同じクラスから作っても、二つの機材は状態を共有しません。

In [ ]:
first = EquipmentItem("E001", "Laptop", "Computer")
second = EquipmentItem("E002", "Projector", "Presentation")
first.loan_to("M014")
print(first.to_record())
print(second.to_record())

## 拒否より前に変更しない

二重貸出を試し、以前の利用者が残ることを確認します。

In [ ]:
try:
    first.loan_to("M021")
except ValueError as error:
    print("REJECTED:", error)
print("Borrower after rejection:", first.borrower_id)
assert first.borrower_id == "M014"

## 四つの境界をテストする

In [ ]:
item2 = EquipmentItem("E010", "Camera", "Media")
assert item2.is_available()
item2.loan_to("M010")
try:
    item2.loan_to("M011")
except ValueError:
    pass
item2.return_item()
try:
    item2.return_item()
except ValueError:
    pass
assert item2.is_available()
print("TRANSITION TESTS PASSED")

## 小さな練習

空白だけのID、空白を含む正常な名称、貸出中の初期状態をそれぞれ試し、どこで正規化・拒否されるか説明してください。